# Making SIMSOPT GPU native: scaled convergence experiment

Select **Runtime > Change runtime type > GPU**, then run all cells. This experiment represents each free-current coordinate in units of 100,000 A, applies the exact chain rule on CPU and GPU, and extends the complete stress-scale L-BFGS-B comparison to 100 iterations. The result is always downloaded, including when the stationary-convergence gate is not yet met.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu"], check=True)

In [ ]:
import shutil

artifact_root = Path("/content/simsopt-scaled-convergence")
if artifact_root.exists():
    shutil.rmtree(artifact_root)
artifact_root.mkdir()
result_file = artifact_root / "stress-scaled-convergence.json"
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/compare_scipy_trajectories.py", "--problem", "stress", "--maxiter", "100", "--current-scale", "100000", "--target-tile-size", "1024", "--source-tile-size", "4320", "--output", str(result_file)], cwd=repo, env=env, check=True)

In [ ]:
import json

result = json.loads(result_file.read_text())
failed = {name: gate for name, gate in result["gates"].items() if not gate["passed"]}
print(json.dumps({"all_gates_passed": result["all_gates_passed"], "coordinate_scaling": result["coordinate_scaling"], "comparison": result["comparison"], "failed_gates": failed}, indent=2))
assert result["objective"]["scope"] == "gpu_native_full_engineering"
assert result["coordinate_scaling"]["current_scale_amperes"] == 100000.0

In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/simsopt-scaled-convergence", "zip", artifact_root)
files.download(archive)